In [1]:
from pyspark.sql import SparkSession

# Para evitar conflito
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Delta_Projeto") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark com Delta iniciado corretamente")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ffdd9385-6dc5-42de-ace5-7cfe81ff53be;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 163ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

Spark com Delta iniciado corretamente


In [2]:
spark.sql("DROP TABLE IF EXISTS delta_table")

import shutil
import os

path = "/app/examples/delta_lake/data/delta_table"

if os.path.exists(path):
    shutil.rmtree(path)

In [3]:
spark.sql("""
CREATE TABLE delta_table (
    palavra STRING,
    valor INT
)
USING DELTA
LOCATION 'file:/app/examples/delta_lake/data/delta_table'
""")

print("Tabela criada")

26/04/25 19:26:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:=========================================>              (37 + 12) / 50]

Tabela criada


In [4]:
spark.sql("""
INSERT INTO delta_table VALUES
('Engenharia', 1),
('Dados', 2),
('Big Data', 3)
""")

spark.sql("SELECT * FROM delta_table").show()

+----------+-----+
|   palavra|valor|
+----------+-----+
|Engenharia|    1|
|  Big Data|    3|
|     Dados|    2|
+----------+-----+



In [5]:
spark.sql("INSERT INTO delta_table VALUES ('Spark', 4)")

DataFrame[]

In [6]:
spark.sql("""
DELETE FROM delta_table
WHERE palavra = 'Engenharia'
""")

DataFrame[num_affected_rows: bigint]

In [7]:
spark.sql("SELECT * FROM delta_table").show()

+--------+-----+
| palavra|valor|
+--------+-----+
|Big Data|    3|
|   Dados|    2|
|   Spark|    4|
+--------+-----+



In [8]:
spark.sql("DESCRIBE HISTORY delta_table").show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      3|2026-04-25 19:26:...|  NULL|    NULL|      DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/3.5....|
|      2|2026-04-25 19:26:...|  NULL|    NULL|       WRITE|{mode -> Append, ...|NULL|    NULL|     NULL|          1|  Serializable|         true|{numFiles -> 1, n...|        NULL|Apache-Spark/3.5.